# DINOv3 

This notebook used DINOv3 in order to retrieve embeddings for sections of the image.  The first code cell tests the model to extract the 
embedding and is basically slightly modified code from the website tutorial on hugging face.

In [7]:
import matplotlib.pyplot as plt
from kitty import depth_read, listPicsWith, togglePath, MatchDepthToCar, KITTY_PATH
from yolo import getEmbedFromResults, getCropsFromResults, getEmbedFromCrops, CLASSES_YOLO, CONFIDENCE_YOLO
from ultralytics import YOLO
from PIL import Image
import numpy as np
import time
import torch

In [ ]:

from transformers import AutoImageProcessor, AutoModel
from transformers.image_utils import load_image

#load a single image
image = load_image("../imgs/image1.jpg")

DINO_MODEL_ID = "facebook/dinov3-vits16-pretrain-lvd1689m"

image_processor = AutoImageProcessor.from_pretrained(DINO_MODEL_ID)
dinov3 = AutoModel.from_pretrained(
    DINO_MODEL_ID,
    dtype=torch.float16,
    device_map="auto"
)

inputs = image_processor(images=image, return_tensors="pt")

dinov3.eval()
with torch.inference_mode():
    outputs = dinov3(**inputs)

#embeddings from output
global_embedding = outputs.pooler_output  # shape: [1, embed_dim]
print("Global embedding shape:", global_embedding.shape)

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 1880.86it/s]


Global embedding shape: torch.Size([1, 384])


In [ ]:
def get_embedding(image):
    """
    Get DINOv3 embeddings from an image or a list of images. 
    Args:
        image: string path or PIL image or list of ready to pricess opened imaged
    """
    #case 1 - image is a path
    if type(image) == str:
        image = load_image(image)
    
    #case 2 is pil image already or compatible with the class
    inputs = image_processor(images=image, return_tensors="pt")

    #retrieve embeddings
    dinov3.eval()
    with torch.inference_mode():
        outputs = dinov3(**inputs)

    return outputs.pooler_output  # final embeddings

In [10]:
images_with_cars = listPicsWith(KITTY_PATH, CLASSES_YOLO, CONFIDENCE_YOLO, True)

model_yolo = YOLO("../build/yolo26n.pt")

results = model_yolo.predict(images_with_cars, conf=CONFIDENCE_YOLO, classes=CLASSES_YOLO)

crops = getCropsFromResults(results)

1000
2011_09_26_drive_0002_sync_image_0000000005_image_02.png
../datasets/depth_selection/val_selection_cropped/image/2011_09_26_drive_0002_sync_image_0000000005_image_02.png

0: 192x640 (no detections), 29.9ms
1: 192x640 (no detections), 29.9ms
2: 192x640 (no detections), 29.9ms
3: 192x640 (no detections), 29.9ms
4: 192x640 (no detections), 29.9ms
5: 192x640 (no detections), 29.9ms
6: 192x640 (no detections), 29.9ms
7: 192x640 (no detections), 29.9ms
8: 192x640 (no detections), 29.9ms
9: 192x640 (no detections), 29.9ms
10: 192x640 (no detections), 29.9ms
11: 192x640 (no detections), 29.9ms
12: 192x640 (no detections), 29.9ms
13: 192x640 (no detections), 29.9ms
14: 192x640 (no detections), 29.9ms
15: 192x640 (no detections), 29.9ms
16: 192x640 (no detections), 29.9ms
17: 192x640 (no detections), 29.9ms
18: 192x640 (no detections), 29.9ms
19: 192x640 (no detections), 29.9ms
20: 192x640 (no detections), 29.9ms
21: 192x640 1 car, 29.9ms
22: 192x640 1 car, 29.9ms
23: 192x640 1 car, 29.9ms


In [12]:
inputs = image_processor(images=crops, return_tensors="pt")

dinov3.eval()
with torch.inference_mode():
    outputs = dinov3(**inputs)

In [13]:
print("Crops embedding shape:", outputs.pooler_output.shape)

Crops embedding shape: torch.Size([2566, 384])
